# Construire et entraîner un mini-décodeur causal

Code et corpus originaux de LLM Academy. Exécuter dans Python avec PyTorch CPU (vérifié avec 2.10.0), pas dans Pyodide. Un bloc, deux têtes, positions apprises, FFN ReLU, pré-normalisation. Ce dispositif n’est pas un modèle factuel ni un GPT complet.


In [ ]:
import math, torch
from torch import nn
torch.manual_seed(7)
torch.set_num_threads(1)
print("PyTorch :", torch.__version__, "CPU")
train_text = ["le chat voit la lune", "le chien voit la lune",
              "le chat voit la lune", "le chien voit le chat"]
validation_text = ["le chat voit le chien", "le chien voit la lune"]
vocab = ["<début>", "<fin>"] + list(dict.fromkeys(" ".join(train_text).split()))
def dataset(lines):
    documents = [[0] + [vocab.index(w) for w in line.split()] + [1] for line in lines]
    assert len({len(doc) for doc in documents}) == 1
    tokens = torch.tensor(documents, dtype=torch.long)
    return tokens[:, :-1], tokens[:, 1:]
x, y = dataset(train_text)
xv, yv = dataset(validation_text)
V, D, MAX_CONTEXT = len(vocab), 16, 8
print("Entrées", tuple(x.shape), "cibles", tuple(y.shape), "V", V)
assert torch.equal(y[:, :-1], x[:, 1:])


## Construire et contrôler avant d’entraîner

Les cibles sont décalées, le masque booléen interdit le futur. Le vocabulaire est construit uniquement sur les documents d’entraînement.


In [ ]:
class MiniGPT(nn.Module):
    """Un bloc pré-normalisé, deux têtes, positions apprises ; sans dropout."""
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(V, D)
        self.position = nn.Embedding(MAX_CONTEXT, D)
        self.ln1 = nn.LayerNorm(D)
        self.attention = nn.MultiheadAttention(D, 2, dropout=0, bias=False, batch_first=True)
        self.ln2 = nn.LayerNorm(D)
        self.ffn = nn.Sequential(nn.Linear(D, 2*D), nn.ReLU(), nn.Linear(2*D, D))
        self.final_norm = nn.LayerNorm(D)
        self.head = nn.Linear(D, V, bias=False)
    def forward(self, ids):
        B, T = ids.shape
        assert 1 <= T <= MAX_CONTEXT
        h = self.embedding(ids) + self.position(torch.arange(T))
        mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
        normalized = self.ln1(h)
        attended, _ = self.attention(normalized, normalized, normalized,
                                    attn_mask=mask, need_weights=False)
        h = h + attended
        h = h + self.ffn(self.ln2(h))
        return self.head(self.final_norm(h))
model = MiniGPT()
criterion = nn.CrossEntropyLoss()
def measured_loss(inputs, targets):
    logits = model(inputs)
    assert logits.shape == (*inputs.shape, V)
    return criterion(logits.reshape(-1, V), targets.reshape(-1))
model.eval()
with torch.no_grad():
    before = model(x).clone()
    changed = x.clone()
    changed[:, -1] = vocab.index("chien")
    after = model(changed)
    torch.testing.assert_close(before[:, :-1], after[:, :-1], rtol=0, atol=1e-6)
print("Formes et causalité : OK ; paramètres :", sum(p.numel() for p in model.parameters()))


## Entraîner puis comparer les pertes

150 pas Adam plein lot ; clipping 1.0. La validation ne contribue à aucun gradient. Les répétitions d’entraînement sont intentionnelles : ce minuscule protocole mesure une expérience, pas une généralisation réelle.


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
history = []
model.eval()
with torch.no_grad():
    initial_train = measured_loss(x, y).item()
for step in range(151):
    model.eval()
    with torch.no_grad():
        train_loss = measured_loss(x, y).item()
        val_loss = measured_loss(xv, yv).item()
    history.append((step, train_loss, val_loss))
    if step % 30 == 0:
        print(f"pas {step:3d} ; entraînement {train_loss:.6f} ; validation {val_loss:.6f}")
    if step == 150:
        break
    model.train()
    optimizer.zero_grad()
    objective = measured_loss(x, y)
    objective.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
assert history[-1][1] < initial_train
# Après entraînement, la causalité doit encore tenir.
model.eval()
with torch.no_grad():
    torch.testing.assert_close(model(x)[:, :-1], model(changed)[:, :-1], rtol=0, atol=1e-6)
print("Entraînement effectif, validation séparée et causalité finale : OK")


## Générer et tracer

RNG PyTorch graine 7, T=0,8, EOS ou 12 nouveaux tokens. Tronquer le contexte réinitialise les positions ; aucun cache KV. Le RNG diffère du navigateur.


In [ ]:
generator = torch.Generator().manual_seed(7)
context = [0, vocab.index("le")]
trace = []
model.eval()
with torch.no_grad():
    for step in range(12):
        cropped = torch.tensor([context[-MAX_CONTEXT:]], dtype=torch.long)
        logits = model(cropped)[0, -1]
        probabilities = torch.softmax(logits / 0.8, dim=-1)
        token = torch.multinomial(probabilities, 1, generator=generator).item()
        trace.append((list(context), token, probabilities[token].item()))
        context.append(token)
        if token == 1:
            break
print("Texte :", " ".join(vocab[i] for i in context))
for context_before, token, probability in trace:
    print("contexte", context_before, "choix", token, "p", round(probability, 6))
assert len(trace) <= 12


## Transfert et sources

Comparer plusieurs corpus et taux avec une initialisation et un budget constants. Garder un test final distinct. Sources : [PyTorch CrossEntropyLoss](https://docs.pytorch.org/docs/2.14/generated/torch.nn.CrossEntropyLoss.html), [Hugging Face causal LM](https://huggingface.co/learn/llm-course/chapter7/6). Les biais et positions apprises de ce notebook sont des choix différents de la miniature TypeScript du module 06.
